# 04 — Fatores da queda e da inoperância em Consignado

## Objetivo

Explicar quais características diferenciam as lojas que perderam produção ou reduziram o valor de Consignado entre **202607 e 202608**, sempre comparando até o **14º dia útil**.

- **Inoperância:** `VLR_CONSIG_ACUM > 0` em julho e igual a zero em agosto.
- **Queda:** redução do `VLR_CONSIG_ACUM` entre julho e agosto nas lojas ativas nos dois meses.
- **Fatores antecedentes:** histórico, posição dos outros produtos em julho e contexto territorial.
- **Co-movimentos:** mudanças simultâneas dos outros produtos entre julho e agosto; são associação, não causa comprovada.

> Os modelos controlam os fatores observados, mas não medem causas operacionais ausentes da base, como visitas, metas, equipe, bloqueios, campanhas ou indisponibilidade sistêmica.


In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score,
    mean_absolute_error, r2_score, roc_auc_score
)
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from db import read_sql

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
warnings.filterwarnings("ignore", category=FutureWarning)

PERIODO_INICIO_HIST = 202602
PERIODO_ANTERIOR = 202607
PERIODO_ATUAL = 202608
DU_CORTE = 14
MIN_BASE_CATEGORIA = 10
RANDOM_STATE = 42

PRODUTOS = {
    "CONTABIL": ("QTD_TRX", "QTD_TRX_ACUM"),
    "CRED_TOTAL": ("CRED_TOTAL", "CRED_TOTAL_ACUM"),
    "CREDITO_JORNADA": ("CREDITO_JORNADA", "CREDITO_JORNADA_ACUM"),
    "PARCELADO": ("VLR_CREDITO_PARCEL", "VLR_CREDITO_PARCEL_ACUM"),
    "CONSIG": ("VLR_CONSIG", "VLR_CONSIG_ACUM"),
    "LIME": ("VLR_LIME", "VLR_LIME_ACUM"),
    "CONTAS": ("CONTAS", "CONTAS_ACUM"),
    "CARTAO_CREDITO": ("QTD_CARTAO_CONTRATADO", "QTD_CARTAO_CONTRATADO_ACUM"),
    "SEGUROS": ("SEG_TOTAL", "SEG_TOTAL_ACUM"),
}


## 1. Extração e auditoria

A produção é enriquecida com hierarquia e população. Os `LEFT JOINs` preservam lojas sem cadastro; a auditoria identifica perdas de cobertura.


In [ ]:
query = f"""
SELECT
    A.*,
    B.CD_MUNIC,
    B.MUNICIPIO,
    B.UF,
    B.DESC_GERENCIA_AREA,
    B.DESC_COORDENACAO,
    B.DESC_SUPERVISAO,
    C.POPULACAO
FROM TESTE..PRODUCAO_DIA_UTIL_BE A
LEFT JOIN DATALAKE..DL_BRADESCO_EXPRESSO B
    ON A.CHAVE_LOJA = B.CHAVE_LOJA
LEFT JOIN IBGE..IBGE_POP C
    ON B.CD_MUNIC = C.COD_UN_REG
WHERE A.PERIODO BETWEEN {PERIODO_INICIO_HIST} AND {PERIODO_ATUAL}
  AND A.QTD_DIA_UTIL_MES <= {DU_CORTE}
"""

print(query)
df = read_sql(query)


In [ ]:
COLUNAS_OBRIGATORIAS = {
    "CHAVE_LOJA", "PERIODO", "QTD_DIA_UTIL_MES",
    "QTD_CONSIG", "VLR_CONSIG", "QTD_CONSIG_ACUM",
    "VLR_CONSIG_ACUM"
}

def auditar_base(df):
    faltantes = sorted(COLUNAS_OBRIGATORIAS - set(df.columns))
    if faltantes:
        raise KeyError(f"Colunas obrigatórias ausentes: {faltantes}")

    chaves = ["CHAVE_LOJA", "PERIODO", "QTD_DIA_UTIL_MES"]
    duplicadas = int(df.duplicated(chaves).sum())
    cobertura = (
        df.groupby("PERIODO")
          .agg(LOJAS=("CHAVE_LOJA", "nunique"),
               ULTIMO_DU=("QTD_DIA_UTIL_MES", "max"))
          .reset_index()
    )

    if duplicadas:
        raise ValueError(
            f"{duplicadas} duplicidades após os JOINs; revise as chaves cadastrais."
        )

    for periodo in [PERIODO_ANTERIOR, PERIODO_ATUAL]:
        du = cobertura.loc[cobertura["PERIODO"] == periodo, "ULTIMO_DU"]
        if du.empty or du.iloc[0] < DU_CORTE:
            raise ValueError(f"Período {periodo} sem cobertura completa do {DU_CORTE}º DU.")

    print(f"Linhas: {len(df):,} | lojas: {df['CHAVE_LOJA'].nunique():,}")
    display(cobertura)
    display(
        df[[c for c in ["UF", "POPULACAO", "DESC_SUPERVISAO"] if c in df]]
        .isna().mean().mul(100).rename("PCT_AUSENTE").to_frame()
    )

auditar_base(df)


## 2. Base analítica e resultados

O universo principal contém somente lojas com valor de Consignado positivo em julho. O critério por quantidade é mantido para reconciliar possíveis divergências cadastrais.


In [ ]:
DIMENSOES = [
    "UF", "DESC_GERENCIA_AREA", "DESC_COORDENACAO",
    "DESC_SUPERVISAO", "POPULACAO"
]

def snapshot_periodo(df, periodo):
    base = df[df["PERIODO"] == periodo].copy()
    return (
        base.sort_values(["CHAVE_LOJA", "QTD_DIA_UTIL_MES"])
            .groupby("CHAVE_LOJA", as_index=False)
            .tail(1)
    )

def criar_coorte(df):
    ant = snapshot_periodo(df, PERIODO_ANTERIOR)
    atual = snapshot_periodo(df, PERIODO_ATUAL)

    acum_produtos = [acum for _, acum in PRODUTOS.values() if acum in df.columns]
    contexto = [c for c in DIMENSOES if c in ant.columns]
    ant_cols = ["CHAVE_LOJA", "QTD_CONSIG_ACUM", "VLR_CONSIG_ACUM"] + acum_produtos + contexto
    ant_cols = list(dict.fromkeys(ant_cols))
    atual_cols = ["CHAVE_LOJA", "QTD_CONSIG_ACUM", "VLR_CONSIG_ACUM"] + acum_produtos
    atual_cols = list(dict.fromkeys(atual_cols))

    a = ant[ant_cols].rename(columns={
        c: f"{c}_ANT" for c in ant_cols if c not in ["CHAVE_LOJA"] + contexto
    })
    b = atual[atual_cols].rename(columns={
        c: f"{c}_ATUAL" for c in atual_cols if c != "CHAVE_LOJA"
    })
    base = a.merge(b, on="CHAVE_LOJA", how="outer")

    medidas = [c for c in base if c.endswith("_ANT") or c.endswith("_ATUAL")]
    base[medidas] = base[medidas].apply(pd.to_numeric, errors="coerce").fillna(0)
    base = base[base["VLR_CONSIG_ACUM_ANT"] > 0].copy()

    base["FLAG_INOPERANTE"] = (base["VLR_CONSIG_ACUM_ATUAL"] <= 0).astype(int)
    base["FLAG_QUEDA"] = (
        (base["VLR_CONSIG_ACUM_ATUAL"] > 0) &
        (base["VLR_CONSIG_ACUM_ATUAL"] < base["VLR_CONSIG_ACUM_ANT"])
    ).astype(int)
    base["VAR_LOG_CONSIG"] = (
        np.log1p(base["VLR_CONSIG_ACUM_ATUAL"]) -
        np.log1p(base["VLR_CONSIG_ACUM_ANT"])
    )
    base["INTENSIDADE_QUEDA"] = -base["VAR_LOG_CONSIG"]
    base["IMPACTO_VALOR"] = (
        base["VLR_CONSIG_ACUM_ANT"] - base["VLR_CONSIG_ACUM_ATUAL"]
    ).clip(lower=0)

    base["DIVERGENCIA_VALOR_QTD"] = (
        (base["VLR_CONSIG_ACUM_ATUAL"] > 0) !=
        (base["QTD_CONSIG_ACUM_ATUAL"] > 0)
    )
    return base

coorte = criar_coorte(df)
display(
    pd.Series({
        "LOJAS_ELEGIVEIS": len(coorte),
        "INOPERANTES": int(coorte["FLAG_INOPERANTE"].sum()),
        "QUEDA_COM_ATIVIDADE": int(coorte["FLAG_QUEDA"].sum()),
        "DIVERGENCIAS_VALOR_QTD": int(coorte["DIVERGENCIA_VALOR_QTD"].sum()),
        "IMPACTO_TOTAL": coorte["IMPACTO_VALOR"].sum(),
    }).to_frame("VALOR")
)


## 3. Fatores antecedentes

As variáveis abaixo terminam em julho. Nenhuma informação de agosto entra nos modelos explicativos.


In [ ]:
def criar_features_historicas(df, lojas):
    hist = df[
        (df["PERIODO"] >= PERIODO_INICIO_HIST) &
        (df["PERIODO"] <= PERIODO_ANTERIOR) &
        (df["CHAVE_LOJA"].isin(lojas))
    ].copy()
    meses = sorted(hist["PERIODO"].unique())

    mensal = (
        hist.groupby(["CHAVE_LOJA", "PERIODO"], as_index=False)
            .agg(QTD_CONSIG_MES=("QTD_CONSIG", "sum"),
                 VLR_CONSIG_MES=("VLR_CONSIG", "sum"),
                 DIAS_CONSIG=("VLR_CONSIG", lambda s: int((s > 0).sum())))
    )
    grade = pd.MultiIndex.from_product(
        [pd.Index(lojas), meses], names=["CHAVE_LOJA", "PERIODO"]
    ).to_frame(index=False)
    mensal = grade.merge(mensal, on=["CHAVE_LOJA", "PERIODO"], how="left").fillna(0)
    mensal["PRODUZIU"] = (mensal["VLR_CONSIG_MES"] > 0).astype(int)

    resumo = mensal.groupby("CHAVE_LOJA").agg(
        MESES_OBSERVADOS=("PERIODO", "nunique"),
        MESES_PROD_CONSIG=("PRODUZIU", "sum"),
        VLR_CONSIG_MEDIA_HIST=("VLR_CONSIG_MES", "mean"),
        VLR_CONSIG_MEDIANA_HIST=("VLR_CONSIG_MES", "median"),
        VLR_CONSIG_DP_HIST=("VLR_CONSIG_MES", "std"),
        QTD_CONSIG_MEDIA_HIST=("QTD_CONSIG_MES", "mean"),
        DIAS_CONSIG_MEDIO=("DIAS_CONSIG", "mean")
    ).reset_index()
    resumo["RECORRENCIA_HIST"] = resumo["MESES_PROD_CONSIG"] / resumo["MESES_OBSERVADOS"]
    resumo["CV_VLR_CONSIG_HIST"] = (
        resumo["VLR_CONSIG_DP_HIST"] / resumo["VLR_CONSIG_MEDIA_HIST"].replace(0, np.nan)
    )
    resumo["TICKET_HIST"] = (
        resumo["VLR_CONSIG_MEDIA_HIST"] / resumo["QTD_CONSIG_MEDIA_HIST"].replace(0, np.nan)
    )

    ordem = {periodo: i for i, periodo in enumerate(meses)}
    mensal["ORDEM_MES"] = mensal["PERIODO"].map(ordem)
    def tendencia(g):
        if len(g) < 2 or g["VLR_CONSIG_MES"].nunique() <= 1:
            return 0.0
        return float(np.polyfit(g["ORDEM_MES"], np.log1p(g["VLR_CONSIG_MES"]), 1)[0])
    tend = mensal.groupby("CHAVE_LOJA").apply(tendencia).rename("TENDENCIA_LOG_CONSIG").reset_index()

    primeiro = (
        hist[hist["VLR_CONSIG"] > 0]
        .groupby(["CHAVE_LOJA", "PERIODO"])["QTD_DIA_UTIL_MES"].min()
        .rename("PRIMEIRO_DU").reset_index()
    )
    ativacao = primeiro.groupby("CHAVE_LOJA").agg(
        PRIMEIRO_DU_MEDIANO=("PRIMEIRO_DU", "median"),
        PCT_INICIO_ATE_DU14=("PRIMEIRO_DU", lambda s: (s <= DU_CORTE).mean() * 100)
    ).reset_index()

    return resumo.merge(tend, on="CHAVE_LOJA", how="left").merge(
        ativacao, on="CHAVE_LOJA", how="left"
    )

feat_hist = criar_features_historicas(df, coorte["CHAVE_LOJA"].unique())
base_modelo = coorte.merge(feat_hist, on="CHAVE_LOJA", how="left")

for produto, (_, acum) in PRODUTOS.items():
    col = f"{acum}_ANT"
    if produto != "CONSIG" and col in base_modelo:
        base_modelo[f"LOG_{produto}_ANT"] = np.log1p(base_modelo[col].clip(lower=0))

if "POPULACAO" in base_modelo:
    base_modelo["LOG_POPULACAO"] = np.log1p(
        pd.to_numeric(base_modelo["POPULACAO"], errors="coerce").clip(lower=0)
    )
    base_modelo["FAIXA_POP"] = pd.cut(
        pd.to_numeric(base_modelo["POPULACAO"], errors="coerce"),
        bins=[-np.inf, 20_000, 50_000, 100_000, 500_000, np.inf],
        labels=["ATÉ 20 MIL", "20–50 MIL", "50–100 MIL", "100–500 MIL", "500 MIL+"]
    )


In [ ]:
NUMERICAS_CANDIDATAS = [
    "VLR_CONSIG_ACUM_ANT", "QTD_CONSIG_ACUM_ANT",
    "RECORRENCIA_HIST", "VLR_CONSIG_MEDIA_HIST",
    "VLR_CONSIG_MEDIANA_HIST", "CV_VLR_CONSIG_HIST",
    "QTD_CONSIG_MEDIA_HIST", "DIAS_CONSIG_MEDIO", "TICKET_HIST",
    "TENDENCIA_LOG_CONSIG", "PRIMEIRO_DU_MEDIANO",
    "PCT_INICIO_ATE_DU14", "LOG_POPULACAO",
] + [f"LOG_{p}_ANT" for p in PRODUTOS if p != "CONSIG"]
CATEGORICAS_CANDIDATAS = [
    "UF", "FAIXA_POP", "DESC_GERENCIA_AREA",
    "DESC_COORDENACAO", "DESC_SUPERVISAO"
]
NUMERICAS = [c for c in NUMERICAS_CANDIDATAS if c in base_modelo]
CATEGORICAS = [c for c in CATEGORICAS_CANDIDATAS if c in base_modelo]
FATORES = NUMERICAS + CATEGORICAS

def criar_preprocessador(numericas, categoricas):
    num = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="infrequent_if_exist",
            min_frequency=MIN_BASE_CATEGORIA
        )),
    ])
    return ColumnTransformer([("num", num, numericas), ("cat", cat, categoricas)])

print(f"Fatores numéricos: {len(NUMERICAS)} | categóricos: {len(CATEGORICAS)}")


## 4. Principais fatores da inoperância

A regressão logística fornece direção; a floresta aleatória e a importância por permutação capturam relações não lineares. A importância é calculada somente na dobra de validação.


In [ ]:
def direcao_numerica(x, y):
    valido = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(valido) < 3 or valido["x"].nunique() < 2:
        return "SEM VARIAÇÃO"
    rho = spearmanr(valido["x"], valido["y"]).statistic
    if abs(rho) < 0.05:
        return "NEUTRA"
    return "AUMENTA O RISCO" if rho > 0 else "REDUZ O RISCO"

def importancia_cv_classificacao(X, y, numericas, categoricas):
    n_min = int(y.value_counts().min())
    if n_min < 2:
        raise ValueError("São necessárias ao menos duas lojas em cada classe.")
    n_splits = min(5, n_min)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    prob = np.zeros(len(X))
    pred = np.zeros(len(X), dtype=int)
    importancias = []

    for treino, teste in cv.split(X, y):
        pipe = Pipeline([
            ("prep", criar_preprocessador(numericas, categoricas)),
            ("modelo", RandomForestClassifier(
                n_estimators=400, min_samples_leaf=8, class_weight="balanced",
                random_state=RANDOM_STATE, n_jobs=-1
            )),
        ])
        pipe.fit(X.iloc[treino], y.iloc[treino])
        prob[teste] = pipe.predict_proba(X.iloc[teste])[:, 1]
        pred[teste] = (prob[teste] >= 0.5).astype(int)
        imp = permutation_importance(
            pipe, X.iloc[teste], y.iloc[teste], scoring="average_precision",
            n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1
        )
        importancias.append(imp.importances_mean)

    imp = np.vstack(importancias)
    ranking = pd.DataFrame({
        "FATOR": X.columns,
        "IMPORTANCIA_MEDIA": imp.mean(axis=0),
        "IMPORTANCIA_DP": imp.std(axis=0),
        "DOBRAS_POSITIVAS": (imp > 0).mean(axis=0),
    })
    ranking["DIRECAO"] = [
        direcao_numerica(X[c], y) if c in numericas else "VARIA POR GRUPO"
        for c in X.columns
    ]
    ranking["ESTAVEL"] = (
        (ranking["IMPORTANCIA_MEDIA"] > 0) &
        (ranking["DOBRAS_POSITIVAS"] >= 0.60)
    )
    metricas = pd.Series({
        "ROC_AUC": roc_auc_score(y, prob),
        "PR_AUC": average_precision_score(y, prob),
        "PR_AUC_BASELINE": y.mean(),
        "BALANCED_ACCURACY": balanced_accuracy_score(y, pred),
    }).to_frame("VALOR")
    return ranking.sort_values("IMPORTANCIA_MEDIA", ascending=False), metricas, prob

X = base_modelo[FATORES].copy()
y_inop = base_modelo["FLAG_INOPERANTE"].astype(int)
ranking_inop, metricas_inop, prob_inop_oof = importancia_cv_classificacao(
    X, y_inop, NUMERICAS, CATEGORICAS
)
base_modelo["PROB_INOPERANCIA_OOF"] = prob_inop_oof
display(metricas_inop)
display(ranking_inop.query("ESTAVEL").head(15))


In [ ]:
modelo_logistico = Pipeline([
    ("prep", criar_preprocessador(NUMERICAS, CATEGORICAS)),
    ("modelo", LogisticRegression(
        max_iter=3000, class_weight="balanced", C=0.5, random_state=RANDOM_STATE
    )),
])
modelo_logistico.fit(X, y_inop)
nomes_transformados = modelo_logistico.named_steps["prep"].get_feature_names_out()
coeficientes_logit = (
    pd.DataFrame({
        "VARIAVEL_TRANSFORMADA": nomes_transformados,
        "COEFICIENTE": modelo_logistico.named_steps["modelo"].coef_[0],
    })
    .assign(ODDS_RATIO=lambda d: np.exp(d["COEFICIENTE"]))
    .sort_values("COEFICIENTE", key=abs, ascending=False)
)
display(coeficientes_logit.head(20))


## 5. Principais fatores da intensidade da queda

O modelo considera apenas lojas ainda ativas em agosto. Valores positivos de `INTENSIDADE_QUEDA` representam redução; valores negativos representam crescimento.


In [ ]:
def importancia_cv_regressao(X, y, numericas, categoricas):
    n_splits = min(5, len(X))
    if n_splits < 2:
        raise ValueError("Base insuficiente para validar o modelo de queda.")
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    previsao = np.zeros(len(X))
    importancias = []

    for treino, teste in cv.split(X):
        pipe = Pipeline([
            ("prep", criar_preprocessador(numericas, categoricas)),
            ("modelo", RandomForestRegressor(
                n_estimators=400, min_samples_leaf=8, max_features=0.7,
                random_state=RANDOM_STATE, n_jobs=-1
            )),
        ])
        pipe.fit(X.iloc[treino], y.iloc[treino])
        previsao[teste] = pipe.predict(X.iloc[teste])
        imp = permutation_importance(
            pipe, X.iloc[teste], y.iloc[teste], scoring="neg_mean_absolute_error",
            n_repeats=15, random_state=RANDOM_STATE, n_jobs=-1
        )
        importancias.append(imp.importances_mean)

    imp = np.vstack(importancias)
    ranking = pd.DataFrame({
        "FATOR": X.columns,
        "IMPORTANCIA_MEDIA": imp.mean(axis=0),
        "IMPORTANCIA_DP": imp.std(axis=0),
        "DOBRAS_POSITIVAS": (imp > 0).mean(axis=0),
    })
    ranking["DIRECAO"] = [
        direcao_numerica(X[c], y) if c in numericas else "VARIA POR GRUPO"
        for c in X.columns
    ]
    ranking["ESTAVEL"] = (
        (ranking["IMPORTANCIA_MEDIA"] > 0) &
        (ranking["DOBRAS_POSITIVAS"] >= 0.60)
    )
    baseline = np.repeat(np.median(y), len(y))
    rho = spearmanr(y, previsao).statistic if len(y) > 2 else np.nan
    metricas = pd.Series({
        "MAE": mean_absolute_error(y, previsao),
        "MAE_BASELINE_MEDIANA": mean_absolute_error(y, baseline),
        "R2": r2_score(y, previsao),
        "SPEARMAN_REAL_PREVISTO": rho,
    }).to_frame("VALOR")
    return ranking.sort_values("IMPORTANCIA_MEDIA", ascending=False), metricas, previsao

ativas = base_modelo[base_modelo["FLAG_INOPERANTE"] == 0].copy().reset_index()
ranking_queda, metricas_queda, pred_queda_oof = importancia_cv_regressao(
    ativas[FATORES], ativas["INTENSIDADE_QUEDA"], NUMERICAS, CATEGORICAS
)
base_modelo["PRED_INTENSIDADE_QUEDA_OOF"] = np.nan
base_modelo.loc[ativas["index"], "PRED_INTENSIDADE_QUEDA_OOF"] = pred_queda_oof
display(metricas_queda)
display(ranking_queda.query("ESTAVEL").head(15))


## 6. Outros produtos: fatores simultâneos

Este bloco responde se a queda de Consignado veio acompanhada por deterioração ampla, preservação da atividade ou mudança de mix. Como agosto faz parte dessas variáveis, elas não entram nos modelos anteriores.


In [ ]:
def analisar_comovimentos(base):
    linhas = []
    for produto, (_, acum) in PRODUTOS.items():
        if produto == "CONSIG":
            continue
        ant, atual = f"{acum}_ANT", f"{acum}_ATUAL"
        if ant not in base or atual not in base:
            continue
        mov = np.log1p(base[atual].clip(lower=0)) - np.log1p(base[ant].clip(lower=0))
        aux = pd.DataFrame({"MOV": mov, "INOPERANTE": base["FLAG_INOPERANTE"]})
        valido = aux.dropna()
        correlacao = np.nan
        if valido["MOV"].nunique() > 1 and valido["INOPERANTE"].nunique() > 1:
            correlacao = spearmanr(valido["MOV"], valido["INOPERANTE"]).statistic
        linhas.append({
            "PRODUTO": produto,
            "MEDIANA_INOPERANTE": aux.loc[aux["INOPERANTE"] == 1, "MOV"].median(),
            "MEDIANA_MANTEVE": aux.loc[aux["INOPERANTE"] == 0, "MOV"].median(),
            "CORRELACAO_COM_INOPERANCIA": correlacao,
        })
    return pd.DataFrame(linhas).sort_values("CORRELACAO_COM_INOPERANCIA", key=abs, ascending=False)

comovimentos = analisar_comovimentos(base_modelo)
display(comovimentos)


## 7. Saída executiva e ranking operacional

O ranking combina impacto financeiro observado, recorrência histórica e risco fora da amostra. Ele prioriza investigação; não substitui validação comercial.


In [ ]:
def fatores_individuais(base, ranking, numericas, top_n=8):
    top = [f for f in ranking.query("ESTAVEL")["FATOR"].head(top_n) if f in numericas]
    medianas = base[top].median(numeric_only=True) if top else pd.Series(dtype=float)
    direcoes = ranking.set_index("FATOR")["DIRECAO"].to_dict()

    def explicar(row):
        fatores = []
        for fator in top:
            valor, mediana = row.get(fator), medianas.get(fator)
            if pd.isna(valor) or pd.isna(mediana) or valor == mediana:
                continue
            risco_alto = (
                (direcoes.get(fator) == "AUMENTA O RISCO" and valor > mediana) or
                (direcoes.get(fator) == "REDUZ O RISCO" and valor < mediana)
            )
            if risco_alto:
                fatores.append(fator)
        return " | ".join(fatores[:3]) or "SEM FATOR NUMÉRICO DOMINANTE"
    return base.apply(explicar, axis=1)

base_modelo["FATORES_LOJA"] = fatores_individuais(
    base_modelo, ranking_inop, NUMERICAS
)
impacto_pct = base_modelo["IMPACTO_VALOR"].rank(pct=True)
rec_pct = base_modelo["RECORRENCIA_HIST"].rank(pct=True).fillna(0)
base_modelo["SCORE_PRIORIDADE"] = (
    60 * impacto_pct +
    25 * rec_pct +
    15 * base_modelo["PROB_INOPERANCIA_OOF"]
)

colunas_ranking = [
    "CHAVE_LOJA", "FLAG_INOPERANTE", "FLAG_QUEDA",
    "VLR_CONSIG_ACUM_ANT", "VLR_CONSIG_ACUM_ATUAL",
    "IMPACTO_VALOR", "RECORRENCIA_HIST",
    "PROB_INOPERANCIA_OOF", "SCORE_PRIORIDADE", "FATORES_LOJA",
] + [c for c in ["UF", "DESC_GERENCIA_AREA", "DESC_COORDENACAO", "DESC_SUPERVISAO"] if c in base_modelo]
ranking_lojas = (
    base_modelo.query("FLAG_INOPERANTE == 1 or FLAG_QUEDA == 1")[colunas_ranking]
    .sort_values("SCORE_PRIORIDADE", ascending=False)
)
display(ranking_lojas.head(50))


In [ ]:
def plot_fatores(ranking, titulo, top_n=12):
    dados = ranking.query("ESTAVEL").head(top_n).sort_values("IMPORTANCIA_MEDIA")
    if dados.empty:
        print(f"{titulo}: nenhum fator estável nas dobras de validação.")
        return
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.barh(dados["FATOR"], dados["IMPORTANCIA_MEDIA"], xerr=dados["IMPORTANCIA_DP"])
    ax.set_title(titulo, loc="left", fontweight="bold")
    ax.set_xlabel("Perda de desempenho ao embaralhar o fator")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_fatores(ranking_inop, "Principais fatores associados à inoperância")
plot_fatores(ranking_queda, "Principais fatores associados à intensidade da queda")


In [ ]:
def resumo_executivo(base, ranking_inop, ranking_queda, comovimentos):
    n = len(base)
    n_inop = int(base["FLAG_INOPERANTE"].sum())
    n_queda = int(base["FLAG_QUEDA"].sum())
    print("RESUMO EXECUTIVO")
    print(f"- Universo elegível: {n:,} lojas.")
    print(f"- Inoperância: {n_inop:,} lojas ({n_inop / n:.1%}).")
    print(f"- Queda com atividade: {n_queda:,} lojas ({n_queda / n:.1%}).")
    print(f"- Impacto observado: {base['IMPACTO_VALOR'].sum():,.2f}.")

    fat_inop = ranking_inop.query("ESTAVEL")["FATOR"].head(5).tolist()
    fat_queda = ranking_queda.query("ESTAVEL")["FATOR"].head(5).tolist()
    print("- Fatores estáveis da inoperância:", ", ".join(fat_inop) or "nenhum")
    print("- Fatores estáveis da queda:", ", ".join(fat_queda) or "nenhum")
    if not comovimentos.empty:
        print("- Maiores co-movimentos com outros produtos:", ", ".join(comovimentos["PRODUTO"].head(3)))
    print("- Leitura: associação estatística; validar causas com dados operacionais e investigação de campo.")

resumo_executivo(base_modelo, ranking_inop, ranking_queda, comovimentos)
